# NLP Final Project – Gemini Flash API
## Real Estate Listing Summarization
**Thành viên:** Nguyễn Bá Học (Gemini Specialist)  
**Model:** `gemini-2.5-flash` (Flash tier – tốc độ cao, chi phí thấp)  
**SDK:** `google-genai` (phiên bản mới nhất)  
**Kỹ thuật:** Few-shot prompting + PII masking

> **Lưu ý:** `gemini-1.5-flash` không còn accessible qua API v1. Dự án sử dụng `gemini-2.5-flash` – cùng Flash tier, hiệu năng tốt hơn.

## Bước 1 – Cài đặt thư viện

In [ ]:
!pip install google-genai pandas openpyxl rouge-score bert-score tqdm python-dotenv

## Bước 2 – Import và cấu hình API

In [ ]:
from google import genai
from google.genai import types
import pandas as pd
import time
import re
import os
from dotenv import load_dotenv
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("Thiếu GEMINI_API_KEY trong file .env")

client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-2.5-flash"

# Config chung: tắt thinking để tốc độ cao như 1.5-flash
GEN_CONFIG = types.GenerateContentConfig(
    temperature=0.3,
    top_p=0.9,
    max_output_tokens=2048,
    thinking_config=types.ThinkingConfig(thinking_budget=0)
)

# Verify model
test_resp = client.models.generate_content(
    model=MODEL_NAME,
    contents="Say OK.",
    config=types.GenerateContentConfig(
        temperature=0, max_output_tokens=50,
        thinking_config=types.ThinkingConfig(thinking_budget=0)
    )
)
print(f"✅ Model {MODEL_NAME} sẵn sàng!")
print(f"   Response test: {test_resp.text.strip()}")

## Bước 3 – Tải dữ liệu

In [ ]:
# Test set – các mẫu cần tóm tắt
test_df = pd.read_csv("data/raw/test.csv", encoding="utf-8-sig")
test_df.columns = test_df.columns.str.strip()
test_df = test_df.dropna(subset=["Mô tả thô"])
test_df = test_df[test_df["Mô tả thô"].str.strip() != ""]
test_df = test_df.reset_index(drop=True)

# Gold samples – few-shot examples và gold reference để đánh giá
sample_df = pd.read_csv("data/gold/sample.csv", encoding="utf-8-sig")
sample_df.columns = sample_df.columns.str.strip()
sample_df = sample_df.dropna(subset=["Mô tả thô", "Mô tả chuẩn"])
sample_df = sample_df[sample_df["Mô tả thô"].str.strip() != ""]
sample_df = sample_df.reset_index(drop=True)

print(f"Test set:    {len(test_df)} mẫu")
print(f"Gold samples: {len(sample_df)} mẫu")
print(f"\nCột test:   {list(test_df.columns)}")
print(f"Cột sample: {list(sample_df.columns)}")
print(f"\n--- Ví dụ 1 gold summary ---")
print(sample_df["Mô tả chuẩn"].iloc[0][:300])

## Bước 4 – Chọn few-shot examples

In [ ]:
# 3 mẫu đa dạng làm few-shot examples:
# 1. Mặt tiền lớn/thương mại (Bình Thạnh, 27.5 tỷ)
# 2. Nhà phố trung – dòng tiền (Quận 3, 8.5 tỷ)
# 3. Nhà nhỏ ở ngay (Quận 10, 7.5 tỷ)
FEW_SHOT_IDS = ["SYS-20264727-938", "SYS-MP756P72-9B", "SYS-MP757WNV-DN"]

few_shot_df = sample_df[sample_df["ID"].isin(FEW_SHOT_IDS)].copy()
eval_df     = sample_df[~sample_df["ID"].isin(FEW_SHOT_IDS)].copy()

# Fallback nếu không tìm đủ theo ID
if len(few_shot_df) < 3:
    print("⚠️ Không tìm đủ ID few-shot → dùng 3 mẫu đầu")
    few_shot_df = sample_df.head(3).copy()
    eval_df     = sample_df.iloc[3:].copy()

print(f"Few-shot examples: {len(few_shot_df)}")
for _, r in few_shot_df.iterrows():
    title = r["Mô tả chuẩn"].split("\n")[0][:80]
    print(f"  [{r['ID']}] {title}")
print(f"\nEval samples (để tính ROUGE/BERTScore): {len(eval_df)}")

## Bước 5 – Xây dựng Few-shot Prompt

In [ ]:
def build_prompt_template(few_shot_df):
    system_instruction = """Bạn là chuyên gia tư vấn bất động sản chuyên nghiệp. Nhiệm vụ của bạn là viết lại tin đăng bất động sản thô thành một mẫu tin đăng chuẩn, súc tích, chuyên nghiệp nhằm thu hút người mua.

Yêu cầu bắt buộc:
- GIỮ NGUYÊN các thông số kỹ thuật: diện tích, số phòng ngủ, số WC, số tầng, giá, hướng nhà, tình trạng pháp lý
- ẨN HOÀN TOÀN: số điện thoại, tên người (chủ/môi giới), số nhà cụ thể, mã nội bộ, emoji
- Sử dụng định dạng chuẩn:
  Dòng 1: [Tên đường] [Quận] - [DT]m2 ([ngang]x[dài]) - [Đặc điểm nổi bật] - [Giá] Tỷ | [Mục đích sử dụng]
  Dòng 2: (trống)
  Dòng 3: [HEADLINE VIẾT HOA - vị trí + đặc điểm + giá]
  Tiếp theo:
  – Vị trí: [mô tả vị trí, tiện ích gần đó, khoảng cách]
  – Hẻm: [mô tả hẻm/đường tiếp cận]
  – Kết cấu: [số tầng, số phòng, nội thất, tình trạng]
  – Diện tích: [diện tích, kích thước ngang x dài, hình dạng đất]
  – Pháp lý sạch, hoàn công đủ, sổ hồng riêng cất két, công chứng ngay.
  – GIÁ: [giá] (TL).
  (Nếu có tiềm năng đầu tư: thêm mục PHÂN TÍCH TƯ DUY ĐẦU TƯ & DÒNG TIỀN)
- Không thêm thông tin không có trong bài gốc\n\n"""

    examples = ""
    for i, (_, row) in enumerate(few_shot_df.iterrows(), 1):
        examples += f"--- VÍ DỤ {i} ---\n"
        examples += f"ĐẦU VÀO:\n{row['Mô tả thô']}\n\n"
        examples += f"ĐẦU RA:\n{row['Mô tả chuẩn']}\n\n"

    template = system_instruction + examples + "--- YÊU CẦU ---\nĐẦU VÀO:\n{raw_text}\n\nĐẦU RA:\n"
    return template


prompt_template = build_prompt_template(few_shot_df)

os.makedirs("prompts", exist_ok=True)
with open("prompts/few_shot.txt", "w", encoding="utf-8") as f:
    f.write(prompt_template)

print(f"Prompt template: {len(prompt_template):,} ký tự")
print("✅ Đã lưu vào prompts/few_shot.txt")

## Bước 6 – Hàm sinh tóm tắt + test 1 mẫu

In [ ]:
def summarize(raw_text, prompt_template, max_retries=3):
    prompt = prompt_template.replace("{raw_text}", raw_text)
    for attempt in range(max_retries):
        try:
            t0 = time.time()
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=GEN_CONFIG
            )
            latency = round(time.time() - t0, 2)
            return {
                "summary": response.text.strip(),
                "latency_s": latency,
                "error": None
            }
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt+1}/{max_retries}: {str(e)[:80]}")
                time.sleep(5)
            else:
                return {"summary": "", "latency_s": 0, "error": str(e)[:200]}


# Test nhanh 1 mẫu đầu tiên
sample_text = test_df["Mô tả thô"].iloc[0]
print("=" * 60)
print("TEST 1 MẪU")
print("=" * 60)
print(f"INPUT (200 ký tự đầu):\n{sample_text[:200]}...\n")

result = summarize(sample_text, prompt_template)

print(f"OUTPUT GEMINI:\n{result['summary']}")
print(f"\nLatency: {result['latency_s']}s")
if result["error"]:
    print(f"Error: {result['error']}")

## Bước 7 – Sinh tóm tắt cho toàn bộ Test Set

In [ ]:
os.makedirs("outputs/summaries", exist_ok=True)
os.makedirs("outputs/metrics", exist_ok=True)

test_results = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test set"):
    out = summarize(row["Mô tả thô"], prompt_template)
    test_results.append({
        "ID": row["ID"],
        "Mô tả thô": row["Mô tả thô"],
        "Mô tả Gemini": out["summary"],
        "Latency (s)": out["latency_s"],
        "Error": out["error"]
    })
    time.sleep(1)  # tránh rate limit

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv("outputs/summaries/gemini_test_results.csv", index=False, encoding="utf-8-sig")

print(f"\n✅ Đã lưu {len(test_results_df)} kết quả")
print(f"Avg latency: {test_results_df['Latency (s)'].mean():.2f}s")
print(f"Max latency: {test_results_df['Latency (s)'].max():.2f}s")
print(f"API errors:  {test_results_df['Error'].notna().sum()} mẫu")

## Bước 8 – Sinh tóm tắt cho Eval Set (để tính ROUGE / BERTScore)

In [ ]:
eval_results = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Eval set"):
    out = summarize(row["Mô tả thô"], prompt_template)
    eval_results.append({
        "ID": row["ID"],
        "Mô tả thô": row["Mô tả thô"],
        "Mô tả chuẩn": row["Mô tả chuẩn"],
        "Mô tả Gemini": out["summary"],
        "Latency (s)": out["latency_s"],
        "Error": out["error"]
    })
    time.sleep(1)

eval_results_df = pd.DataFrame(eval_results)
eval_results_df.to_csv("outputs/summaries/gemini_eval_results.csv", index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu {len(eval_results_df)} eval results")

## Bước 9 – Đánh giá ROUGE-L

In [ ]:
from rouge_score import rouge_scorer as rs

scorer = rs.RougeScorer(["rougeL"], use_stemmer=False)

def compute_rouge_l(pred, ref):
    if not pred or not ref or not str(pred).strip() or not str(ref).strip():
        return 0.0
    return round(scorer.score(str(ref), str(pred))["rougeL"].fmeasure, 4)

eval_results_df["ROUGE-L"] = eval_results_df.apply(
    lambda r: compute_rouge_l(r["Mô tả Gemini"], r["Mô tả chuẩn"]), axis=1
)

print("ROUGE-L per sample:")
print(eval_results_df[["ID", "ROUGE-L"]].to_string(index=False))
print(f"\nMean ROUGE-L: {eval_results_df['ROUGE-L'].mean():.4f}")
print(f"Std  ROUGE-L: {eval_results_df['ROUGE-L'].std():.4f}")

## Bước 10 – Đánh giá BERTScore

In [ ]:
from bert_score import score as bert_score_fn

valid_mask = (
    eval_results_df["Mô tả Gemini"].notna() &
    eval_results_df["Mô tả chuẩn"].notna() &
    (eval_results_df["Mô tả Gemini"].str.strip() != "") &
    (eval_results_df["Mô tả chuẩn"].str.strip() != "")
)
valid_df = eval_results_df[valid_mask].copy()

preds = valid_df["Mô tả Gemini"].tolist()
refs  = valid_df["Mô tả chuẩn"].tolist()

# bert-base-multilingual-cased – không cần download thêm, hỗ trợ tiếng Việt
P, R, F1 = bert_score_fn(
    preds, refs,
    lang="vi",
    model_type="bert-base-multilingual-cased",
    verbose=True
)

eval_results_df.loc[valid_mask, "BERTScore-P"]  = [round(p.item(), 4) for p in P]
eval_results_df.loc[valid_mask, "BERTScore-R"]  = [round(r.item(), 4) for r in R]
eval_results_df.loc[valid_mask, "BERTScore-F1"] = [round(f.item(), 4) for f in F1]

print("BERTScore-F1 per sample:")
print(eval_results_df[["ID", "BERTScore-F1"]].to_string(index=False))
print(f"\nMean BERTScore-F1: {eval_results_df['BERTScore-F1'].mean():.4f}")

eval_results_df.to_csv("outputs/metrics/gemini_metrics.csv", index=False, encoding="utf-8-sig")
print("✅ Đã lưu → outputs/metrics/gemini_metrics.csv")

## Bước 11 – Kiểm tra rò rỉ PII (thông tin nhạy cảm)

In [ ]:
def check_pii_leakage(text):
    if not text or not str(text).strip():
        return {"phones_leaked": [], "addresses_leaked": [], "is_clean": True}
    text = str(text)
    phones = re.findall(r"(0|\+84)[0-9]{8,9}", text)
    # Số nhà cụ thể: "số 12", "12/3A", "12A/"
    addrs  = re.findall(r"\bsố\s+\d+\b|\b\d+[/\\]\d+[A-Za-z]?", text, re.IGNORECASE)
    return {
        "phones_leaked": phones,
        "addresses_leaked": addrs,
        "is_clean": len(phones) == 0 and len(addrs) == 0
    }

test_results_df["PII_Check"] = test_results_df["Mô tả Gemini"].apply(check_pii_leakage)
test_results_df["PII_Clean"] = test_results_df["PII_Check"].apply(lambda x: x["is_clean"])

leaked = test_results_df[~test_results_df["PII_Clean"]]
print(f"Tổng mẫu:         {len(test_results_df)}")
print(f"PII sạch:         {test_results_df['PII_Clean'].sum()}")
print(f"Có rò rỉ PII:     {len(leaked)}")

if len(leaked) > 0:
    print("\n⚠️ Mẫu cần xem lại:")
    for _, r in leaked.iterrows():
        print(f"  ID: {r['ID']} | {r['PII_Check']}")
else:
    print("\n✅ Tất cả mẫu đã ẩn thông tin nhạy cảm!")

## Bước 12 – Tổng hợp kết quả

In [ ]:
print("=" * 60)
print("KẾT QUẢ TỔNG HỢP – GEMINI 2.5 FLASH")
print("=" * 60)

print(f"\n📊 Test Set: {len(test_results_df)} mẫu")
print(f"   Avg Latency:    {test_results_df['Latency (s)'].mean():.2f}s")
print(f"   Max Latency:    {test_results_df['Latency (s)'].max():.2f}s")
print(f"   PII Clean Rate: {test_results_df['PII_Clean'].mean():.1%}")
print(f"   API Errors:     {test_results_df['Error'].notna().sum()}")

if "ROUGE-L" in eval_results_df.columns:
    print(f"\n📊 Eval Set: {len(eval_results_df)} mẫu (vs gold reference)")
    print(f"   Avg ROUGE-L:      {eval_results_df['ROUGE-L'].mean():.4f}")
    if "BERTScore-F1" in eval_results_df.columns:
        print(f"   Avg BERTScore-F1: {eval_results_df['BERTScore-F1'].mean():.4f}")

rouge_val = f"{eval_results_df['ROUGE-L'].mean():.4f}" if "ROUGE-L" in eval_results_df.columns else "(chờ eval)"
bert_val  = f"{eval_results_df['BERTScore-F1'].mean():.4f}" if "BERTScore-F1" in eval_results_df.columns else "(chờ eval)"

print("\n" + "=" * 60)
print("Bảng so sánh (điền kết quả các model khác sau):")
print("-" * 60)
comparison = pd.DataFrame({
    "Model": ["TextRank (baseline)", "GPT-4o", "Gemini 2.5 Flash", "ViT5 fine-tuned", "PhoBERT fine-tuned"],
    "ROUGE-L": ["", "", rouge_val, "", ""],
    "BERTScore-F1": ["", "", bert_val, "", ""],
    "Latency (s)": ["", "", f"{test_results_df['Latency (s)'].mean():.2f}", "", ""],
    "Expert Score": ["", "", "(chờ chuyên gia)", "", ""]
})
print(comparison.to_string(index=False))

## Bước 13 – Xuất file cho chuyên gia chấm điểm

In [ ]:
export_df = test_results_df[["ID", "Mô tả thô", "Mô tả Gemini"]].copy()
export_df["TC1 - Chính xác thông số (1-5)"]   = ""
export_df["TC2 - Hấp dẫn người mua (1-5)"]    = ""
export_df["TC3 - Văn phong chuyên nghiệp (1-5)"] = ""
export_df["TC4 - Bảo mật thông tin (1-5)"]    = ""
export_df["Điểm tổng (1-5)"]                  = ""
export_df["Nhận xét chuyên gia"]              = ""

expert_path = "outputs/summaries/gemini_for_expert_review.xlsx"
export_df.to_excel(expert_path, index=False)
print(f"✅ Xuất {len(export_df)} mẫu → {expert_path}")
print("\nGửi file này cho chuyên gia BĐS để chấm điểm.")

## Bước 14 – Xem thử 3 kết quả mẫu

In [ ]:
for i, row in test_results_df.head(3).iterrows():
    print(f"{'='*65}")
    print(f"MẪU {i+1} | ID: {row['ID']} | Latency: {row['Latency (s)']}s | PII: {'✅' if row['PII_Clean'] else '⚠️'}")
    print(f"{'='*65}")
    print(f"INPUT (200 ký tự đầu):\n{str(row['Mô tả thô'])[:200]}...")
    print(f"\nOUTPUT GEMINI:\n{row['Mô tả Gemini']}")
    print()